In [7]:
# Load Emme trip tables for a time period
# Use Emme API
import inro.emme.desktop.app as app
import inro.modeller as _m
import pandas as pd
import numpy as np

In [ ]:

filepath = f"C:/workspace/sc_main_RTP_2023/projects/7to8/7to8.emp"

desktop = app.start_dedicated(True, "cth", filepath)
data_explorer = desktop.data_explorer()
m = _m.Modeller(desktop) 
database_names = [
            database.title() for database in data_explorer.databases()
        ]

database_title = "7to8"
scenario_id = "1002"

if database_title in database_names:
    database_index = database_names.index(database_title)
    database = data_explorer.databases()[database_index]
    database.open()
    bank = m.emmebank

current_scenario = list(bank.scenarios())[0]
zonesDim = len(current_scenario.zone_numbers)
zones = current_scenario.zone_numbers

# load zones into a NumpyArray to index trips otaz and dtaz

# Create a dictionary lookup where key is the taz id and value is it's numpy index.
indexToZone = dict((index, zone) for index, zone in enumerate(zones))

In [11]:
matrix_name = "sov_inc2d"
print(f"Processing matrix: {matrix_name}")
matrix_id = bank.matrix(matrix_name).id
emme_matrix = bank.matrix(matrix_id)
matrix_data = emme_matrix.get_data()

# Sum columns (destinations) of the matrix
np_matrix = matrix_data.to_numpy()
# dest_sums = np_matrix.sum(axis=0)  # one value per destination TAZ index

# Use indexToZone to create a column and index labels for a dataframe
dest_index_to_zone = {index: indexToZone[index] for index in range(zonesDim)}
df = pd.DataFrame(np_matrix, columns=[dest_index_to_zone[i] for i in range(zonesDim)])
df.index = [indexToZone[i] for i in range(zonesDim)]

Processing matrix: sov_inc2d


In [19]:
df.to_csv(r"R:\e2projects_two\data_requests\2026\regional_EVSE_plan\EVSE_updated_data\sov_distance_skim.csv")

In [ ]:
filepath = f"C:/workspace/sc_main_RTP_2023/projects/LoadTripTables/LoadTripTables.emp"

desktop = app.start_dedicated(True, "cth", filepath)
data_explorer = desktop.data_explorer()
m = _m.Modeller(desktop) 
database_names = [
            database.title() for database in data_explorer.databases()
        ]



In [4]:
database_title = "daily"
scenario_id = "1002"

if database_title in database_names:
    database_index = database_names.index(database_title)
    database = data_explorer.databases()[database_index]
    database.open()
    bank = m.emmebank

current_scenario = list(bank.scenarios())[0]
zonesDim = len(current_scenario.zone_numbers)
zones = current_scenario.zone_numbers

# load zones into a NumpyArray to index trips otaz and dtaz

# Create a dictionary lookup where key is the taz id and value is it's numpy index.
indexToZone = dict((index, zone) for index, zone in enumerate(zones))

In [19]:
# Get demand
sov_list = ["sov_inc1", "sov_inc2", "sov_inc3"]
hov2_list = ["hov2_inc1", "hov2_inc2", "hov2_inc3"]
hov3_list = ["hov3_inc1", "hov3_inc2", "hov3_inc3"]
tnc_list = ["tnc_inc1", "tnc_inc2", "tnc_inc3"]
truck_list = ["medium_truck", "heavy_truck"]

results_df = pd.DataFrame()

matrix_data = np.zeros((zonesDim, zonesDim))  # Initialize an empty matrix to accumulate the data

for matrix_name in sov_list + hov2_list + hov3_list + tnc_list:
    print(f"Processing matrix: {matrix_name}")
    matrix_id = bank.matrix(matrix_name).id
    emme_matrix = bank.matrix(matrix_id)
    matrix_data = matrix_data + emme_matrix.get_data().to_numpy() 

dest_index_to_zone = {index: indexToZone[index] for index in range(zonesDim)}
df_vehicle = pd.DataFrame(matrix_data, columns=[dest_index_to_zone[i] for i in range(zonesDim)])
df_vehicle.index = [indexToZone[i] for i in range(zonesDim)]

truck_matrix_data = np.zeros((zonesDim, zonesDim))  # Initialize an empty matrix to accumulate the data
for matrix_name in truck_list:
    print(f"Processing matrix: {matrix_name}")
    matrix_id = bank.matrix(matrix_name).id
    emme_matrix = bank.matrix(matrix_id)
    truck_matrix_data = truck_matrix_data + emme_matrix.get_data().to_numpy() 

dest_index_to_zone = {index: indexToZone[index] for index in range(zonesDim)}
df_truck = pd.DataFrame(truck_matrix_data, columns=[dest_index_to_zone[i] for i in range(zonesDim)])
df_truck.index = [indexToZone[i] for i in range(zonesDim)]

Processing matrix: sov_inc1
Processing matrix: sov_inc2
Processing matrix: sov_inc3
Processing matrix: hov2_inc1
Processing matrix: hov2_inc2
Processing matrix: hov2_inc3
Processing matrix: hov3_inc1
Processing matrix: hov3_inc2
Processing matrix: hov3_inc3
Processing matrix: tnc_inc1
Processing matrix: tnc_inc2
Processing matrix: tnc_inc3
Processing matrix: medium_truck
Processing matrix: heavy_truck


In [20]:
df_truck.sum().sum()

453253.03904476017

In [21]:
df_vehicle.sum().sum()

10884403.432628296

In [24]:
df_truck.to_csv(r"R:\e2projects_two\data_requests\2026\regional_EVSE_plan\EVSE_updated_data\truck_daily_trip_table.csv")
df_vehicle.to_csv(r"R:\e2projects_two\data_requests\2026\regional_EVSE_plan\EVSE_updated_data\vehicle_daily_trip_table.csv")